# Scipy Precipitation and Sea Level Pressure Features Detection and Tracking Algorithm

## Overview
Previously, you have learned how to detect precipitation and sea level pressure (SLP) features seperately. However, what if you wanted to plot the sea level pressure and precipitation features together, and plot them together in one plot? In this notebook, we will apply the previous algorithms to create plots analyzing SLP and preicpitation features using Scipy's label function.

## Prerequisites
| Concepts | Importance | Notes |
| --- | --- | --- |
| [xarray](https://foundations.projectpythia.org/core/xarray/) | Necessary | |
| [matplotlib](https://foundations.projectpythia.org/core/matplotlib/matplotlib-basics/) | Necessary | |
| [Cartopy](https://foundations.projectpythia.org/core/cartopy/cartopy) | Necessary | Needed to plot geospatial data|
| [Scipy Precip Detection and Tracking Algorithm](https://projectpythia.org/feature-tracking-cookbook/notebooks/scipy-precip/) | Necessary | |
| [Scipy SLP Detection and Tracking Algorithm](https://projectpythia.org/feature-tracking-cookbook/notebooks/slp-scipy/) | Necessary | |
| [scipy-image](https://docs.scipy.org/doc/scipy/reference/ndimage.html) | Useful | Useful reference for Scipy's label function |
| [IPython](https://ipython.readthedocs.io/en/latest/interactive/plotting.html) | Useful | For inline animation|

## Import Packages

In [ ]:
import xarray as xr
from scipy.ndimage import label, generate_binary_structure
import matplotlib.pyplot as plt
from scipy.ndimage import minimum_filter
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

import matplotlib.colors as mcolors
from matplotlib.cm import ScalarMappable
import pandas as pd
import math

Let's create a function that handles events that cross over the international dateline.

In [ ]:
def crosses_dateline(labeled_arr, resolution):
    #Shift 180 degrees
    shifted_arr = np.roll(labeled_arr, 180*resolution, axis = 1)

    #Examine a 3x3 region centered at the dateline
    for lat in range(0, len(shifted_arr) - 1):
    	sample_arr = shifted_arr[lat:lat+2, 179:181]

    	top_left = sample_arr[0, 0]
    	top_right = sample_arr[0, 1]
    	bottom_right = sample_arr[1, 1]
    	bottom_left = sample_arr[1, 0]
		
    	#Compare top left with position next to it
    	if(top_left != 0 and top_right != 0):
    		if(top_left != top_right):
    			num_to_change = top_left
    			shifted_arr = np.where(shifted_arr != num_to_change, shifted_arr, top_right)
    			sample_arr = shifted_arr[lat:lat+2, 179:181]

    	#Compare top left with position diagonal to it
    	if(top_left != 0 and bottom_right != 0):
    		if(top_left != bottom_right):
    			num_to_change = top_left
    			shifted_arr = np.where(shifted_arr != num_to_change, shifted_arr, bottom_right)
    			sample_arr = shifted_arr[lat:lat+2, 179:181]
		
    	#Compate bottom left with position next to it
    	if(bottom_left != 0 and bottom_right != 0):
    		if(bottom_left != bottom_right):
    			num_to_change = bottom_left
    			shifted_arr = np.where(shifted_arr != num_to_change, shifted_arr, bottom_right)
    			sample_arr = shifted_arr[lat:lat+2, 179:181]
    			#print('Changed arr bot left, bot right', sample_arr)
    	
    	#Compare bottom left with position diagonal to it
    	if(bottom_left != 0 and top_right != 0):
    		if(bottom_left != top_right):
    			num_to_change = bottom_left
    			shifted_arr = np.where(shifted_arr != num_to_change, shifted_arr, top_right)
    			sample_arr = shifted_arr[lat:lat+2, 179:181]
    		
    return np.roll(shifted_arr, 180, axis = 1)
    
    

# Basic Set-Up

We have two previous algorithms that are useful to detect precipitation and SLP features. Let's create a plot that plots both of these features together, which should hopefully reveal some characteritsics about extratropical cyclones (ETCs).

Before we do that though, we need to implement both the precipitation detection function and the SLP detection function.

Load in both datasets using xarray:

In [ ]:
precip_data = xr.open_dataset('data/precip_ex.nc')
slp_data = xr.open_dataset('../notebooks/data/SLP_ex.nc')

The data contained within `slp_data` is in pascals, however atmospheric scientists commonly use hectopascals. We will therefore need to convert the array from pascals into hectopascals by dividing by 100. 

In [ ]:
slp = slp_data['SLP']/100

Recall that although we have global data for `slp_data`, we only have northern hemisphere (specifically between 20°N and 80°N) data for `precip_data`. Therefore, we need to filter out the `slp_data` so that it only includes the same data points that are contained within the `precip_data`. 

In [ ]:
lat_vals = slp['latitude']
slp_nh_filtered = slp.where(lat_vals >= 20, drop=True)
slp_nh_filtered = slp_nh_filtered.where(lat_vals <= 80, drop=True)

Let's create a 2-d grid for each of our datasets. Unfortunately, the two datasets do not have similar resolutions. The resolution of `slp_data` is 1° x 1°, while the resolution for `precip_data` is 0.25° x 0.25°. This therefore requires us to create two different grids for each of our datasets. 

In [ ]:
lon2d_slp, lat2d_slp = np.meshgrid(slp_nh_filtered['longitude'].values, slp_nh_filtered['latitude'].values)
lon2d_precip, lat2d_precip = np.meshgrid(precip_data['longitude'].values, precip_data['latitude'].values)

# Scipy Precipitation and SLP Detection Algorithm

In order to identify slp minima, we need to set a threshold. We'll say that any feature with a pressure below 1000hPa is a feature that we are interested in tracking for the SLP dataset. 

Similarly, we'll set up a threshold for minimum precipitation as well of 0.25 mm/hr.

In [ ]:
slp_threshold = 1000
slp_1000 = np.where(slp_nh_filtered < slp_threshold, slp_nh_filtered, 0)

precip_threshold = 0.25
precip_25 = np.where(precip_data['ETC_tp'] > precip_threshold, precip_data['ETC_tp'], 0)

Let's now run our algorithm to detect features that we have previously built in the Scipy SLP Detection and Tracking Algorithm notebook. We will first focus on a single time step.

In [ ]:
s = generate_binary_structure(2,2)
def detect_feature(data, time_index, resolution):
    labeled_data_field, num_features = label(data[time_index], s)
    labeled_data_field = crosses_dateline(labeled_data_field, resolution)
    return labeled_data_field

Now that we have our function, let's run this function on both `preicp_data` and `slp_data` for the first time index.

In [ ]:
precip_features = detect_feature(precip_25, 0, 0.25)
slp_features = detect_feature(slp_1000, 0, 1)

Let's go ahead and plot both the `precip_features` and `slp_features`.

In [ ]:
plt.figure(figsize=(15,8))
ax = plt.axes(projection=ccrs.PlateCarree())
plt.contour(lon2d_slp, lat2d_slp, slp_features, cmap='tab20b')
## For precip features, let's convert anything associated with feature 0 (i.e. no rain) into np.nan so it doesn't show up as contoured.
precip_features_nan = np.where(precip_features == 0, np.nan, precip_features)
plt.contourf(lon2d_precip, lat2d_precip, precip_features_nan, cmap='tab20c')


ax.add_feature(cfeature.LAND)
ax.add_feature(cfeature.BORDERS, edgecolor='black')
ax.add_feature(cfeature.STATES, edgecolor='black')
ax.add_feature(cfeature.COASTLINE, edgecolor='black')

ax.set_title('Precip and SLP Features at Time Index 0 Overlaid')
ax.gridlines(draw_labels=True)

In this map, slp features are contoured and precipitation features are in filled contours. We can see in general that most slp features are associated with some precipitation features. However, in the northern latitudes such as near the Hudson Bay and off the coast of Alaska there are some exceptions. The reverse is not true. Not every precipitation feature is associated with a low slp feature, especially in regions closer to the equator. This indicates that most of the slp regions identified are associated with some sort of `ETC`, while the same is not true for precipitation regions.

Let's create an animation showing the evolution of this.

In [ ]:
## Let's create a function to write the figures necessary for the animation.


fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()})

def func(i):
    ax.clear()

    precip_features = detect_feature(precip_25, i, 0.25)
    slp_features = detect_feature(slp_1000, i, 1)
    plt.contour(lon2d_slp, lat2d_slp, slp_features, cmap='tab20b')
    ## For precip features, let's convert anything associated with feature 0 (i.e. no rain) into np.nan so it doesn't show up as contoured.
    precip_features_nan = np.where(precip_features == 0, np.nan, precip_features)
    plt.contourf(lon2d_precip, lat2d_precip, precip_features_nan, cmap='tab20c')

    ax.add_feature(cfeature.LAND)
    ax.add_feature(cfeature.BORDERS, edgecolor='black')
    ax.add_feature(cfeature.STATES, edgecolor='black')

    ax.set_extent([-180, 180, 20, 80])
    
    ax.add_feature(cfeature.COASTLINE, edgecolor='black')

    ax.set_title('Precip and SLP Features Overlaid at Time Index ' + str(i), fontsize=8)
    
    provinces = cfeature.NaturalEarthFeature(
        category='cultural',
        name='admin_1_states_provinces_lines',
        scale='50m',
        facecolor='none',
        edgecolor='black'
    )
    ax.add_feature(provinces)
    ax.gridlines(draw_labels=True)
    plt.tight_layout()
    return fig,


anim = FuncAnimation(fig, func, frames=12, interval=300, blit=False)
HTML(anim.to_jshtml())
# anim.save('my_animation.gif', writer='pillow', fps=2)

## Getting Precipitation Rate and Minimum SLP for Each Feature

Like we did in previous notebooks, let's take the functions that were previously defined for minimum slp and precipitation rate to make a plot.

In [ ]:
minima = minimum_filter(slp_nh_filtered, 5, mode=['nearest', 'wrap'], axes = [1, 2])
minima_3 = (slp_nh_filtered == minima)
min_times, min_lats, min_lons = np.where(1 == minima_3)

Now let's go ahead and make an animation of this time series!

In [ ]:
## Let's create a function to write the figures necessary for the animation.


fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()})

## Creating a colorbar across all frames.
vmin = 0
vmax = 2
levels = np.linspace(vmin, vmax, 10)
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
cmap='Greens'

sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, pad=0.02, shrink=0.4)
cbar.set_label("Precipitation Rate [mm/hr]")

def func(i):
    ax.clear()

    where_time = np.where(min_times == i)
    timed_lats = min_lats[where_time]
    timed_lons = min_lons[where_time]

    precip_data_filtered = np.where(precip_data['ETC_tp'][i] > 0, precip_data['ETC_tp'][i], np.nan)

    ax.contourf(precip_data['longitude'], precip_data['latitude'], precip_data_filtered, vmin=0, vmax=2, cmap='Greens')

    ax.scatter(slp_nh_filtered[i].longitude[timed_lons], slp_nh_filtered[i].latitude[timed_lats])

    ax.add_feature(cfeature.LAND)
    ax.add_feature(cfeature.BORDERS, edgecolor='black')
    ax.add_feature(cfeature.STATES, edgecolor='black')

    ax.set_extent([-180, 180, 20, 80])
    
    ax.add_feature(cfeature.COASTLINE, edgecolor='black')

    ax.set_title('Precip and SLP Features Overlaid at Time Index ' + str(i), fontsize=8)
    
    provinces = cfeature.NaturalEarthFeature(
        category='cultural',
        name='admin_1_states_provinces_lines',
        scale='50m',
        facecolor='none',
        edgecolor='black'
    )
    ax.add_feature(provinces)
    ax.gridlines(draw_labels=True)
    plt.tight_layout()
    return fig,


anim = FuncAnimation(fig, func, frames=12, interval=300, blit=False)
HTML(anim.to_jshtml())
# anim.save('my_animation.gif', writer='pillow', fps=2)

It's important to note that in this diagram, any minima in sea level pressure is shown via the dots, not just those below 1000hPa. Therefore, there are a lot more dots on this map than features identified in the last image. Precipitation rate is in filled contours underneath the dots. These are filtered to only include areas where the precipitation rate is above 0mm/hr. 

We can see a consistent pattern here. Anywhere there is precipitation, there is at least some local minima in SLP. We previously saw in the last example that the low pressure system doesn't have to be below 1000hPa, but in every precipitation anomaly is at the very least associated with a local minima in SLP. 